## Sample Write Delta table on Minio

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")

NameError: name 'load_dotenv' is not defined

In [ ]:
conf = SparkConf()

conf.setAppName("Example write delta table on minIO s3") # nome da aplicação Spark, útil para logs
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", f"{MINIO_USER}") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", f"{MINIO_PASSWORD}") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True)
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()


## Add data for Dataframe

In [5]:
exampleData = [("Rodrigo", "Maturino", "M", 4000),
            ("Rafael", "Souza", "M", 4500),
            ("Luigi", "Bispo", "M", 6000),
            ("Eduarda", "Santos", "F", 5000)]

## Add a schema for DataFrame

In [4]:
schema = StructType([
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("salary", IntegerType(), True)
        ])

In [6]:
dataFrame = spark.createDataFrame(data=exampleData, schema=schema)

In [7]:
dataFrame.show()

+----------+---------+------+------+
|first_name|last_name|Gender|salary|
+----------+---------+------+------+
|   Rodrigo| Maturino|     M|  4000|
|    Rafael|    Souza|     M|  4500|
|     Luigi|    Bispo|     M|  6000|
|   Eduarda|   Santos|     F|  5000|
+----------+---------+------+------+



## Saving the file to the MinIO S3 bronze bucket in Delta format

In [9]:
dataFrame.write.format("delta").mode("append").save("s3a://bronze/delta_example")